# DEEZER API PULL

11/5/26 script to pull deezer data from their public API.
- have genre
- can pull top 50 artists in each genre (url format: base + /genre + /genre_id + /artists)
- goal: dataframe of top 50 artists per genre

functions:
- get genre
- get artists from genre
- validation
- make dataframe

ok so the top 50 doesnt work well, what we're gonna do is get the top x tracks across all genres, plays, and get the top artist by plays of each days, top x in most genres, yesterdays, past week, past month (when available). lets also make this just deezer

In [1]:
import argparse
import requests
import pandas as pd
import time
import boto3

In [ ]:
MAX_RETRIES = 3 # still need to implement this
url = 'https://api.deezer.com'
session = requests.Session()

In [ ]:
# error class so we know where the error comes from further down the line/when the script is run
class DeezerError(Exception):
    pass

# DEFINE FUNCTIONS

In [ ]:
def get_genre_data() -> pd.DataFrame:
    try:
        response = session.get(f'{url}/genre', timeout=15)
        response.raise_for_status()
        body = response.json() # the whole response
        if "error" in body:
            raise DeezerError('error in genres data body')
        df = pd.DataFrame(body["data"])[['id', 'name']]
        return df[df['id'] != 0] # no 'all' genre
    except requests.RequestException as e:
        print("request failed:", e)
        raise
    except DeezerError as e:
        print("deezer error:", e)
        raise

genres = get_genre_data()

In [40]:
def get_top_tracks() -> pd.DataFrame:
    try:
        result = []
        for genre_id, genre in zip(genres['id'], genres['name']):
            response = session.get(
                f'{url}/chart/{genre_id}/tracks',
                params={'limit': 100},
                timeout=15,)
            response.raise_for_status()
            body = response.json() # the whole response
            if "error" in body:
                raise DeezerError('error in artists data body')
            # dataframe transformation
            df = pd.DataFrame(body['data'])
            df['Genre'] = genre # add in genre data
            result.append(df)
            # print(len(result))
            # time.sleep(.5)
        return pd.concat(result, ignore_index=True)
    except requests.RequestException as e:
            print("request failed:", e)
            raise
    except DeezerError as e:
            print("deezer error:", e)
            raise

top_tracks = get_top_tracks()

In [ ]:
print(top_tracks.shape)
top_tracks.sample(1)

(2700, 17)


,id,title,title_short,title_version,link,duration,rank,explicit_lyrics,explicit_content_lyrics,explicit_content_cover,preview,md5_image,position,artist,album,type,Genre
1759,3819914,Before I Forget,Before I Forget,,https://www.deezer.com/track/3819914,278,802756,False,0,0,https://cdnt-preview.dzcdn.net/api/1/1/a/d/6/0...,35b093d22fe1539003d5d18dd8f309eb,60,"{'id': 117, 'name': 'Slipknot', 'link': 'https...","{'id': 356130, 'title': 'Vol. 3: The Sublimina...",track,Metal


In [ ]:
# expanding out the artist column from a dictionary to separate columns to be added back to the tracks dataframe
artist_info_df = pd.json_normalize(top_tracks['artist'])
artist_info_df = artist_info_df.rename(columns={'id':'artist_id',
                                                'name':'artist_name',
                                                'link':'artist_link',
                                                'picture':'artist_picture',
                                                'tracklist':'top_tracklist'}, errors='ignore')[['artist_id','artist_name','artist_link','artist_picture','top_tracklist']]
artist_info_df.sample(1)

,artist_id,artist_name,artist_link,artist_picture,top_tracklist
1206,9626504,Luke Combs,https://www.deezer.com/artist/9626504,https://api.deezer.com/artist/9626504/image,https://api.deezer.com/artist/9626504/top?limi...


In [ ]:
album_info_df = pd.json_normalize(top_tracks['album'])
album_info_df = album_info_df.rename(columns={'id':'album_id',
                                              'title':'album_title',
                                              'cover':'album_cover',
                                              'tracklist':'album_tracklist'}, errors='ignore')[['album_id','album_title','album_cover','album_tracklist']]
album_info_df.sample(1)

,album_id,album_title,album_cover,album_tracklist
2158,8088474,King of the Delta Blues SIngers,https://api.deezer.com/album/8088474/image,https://api.deezer.com/album/8088474/tracks


In [69]:
top_tracks = pd.concat([top_tracks, artist_info_df, album_info_df], axis=1).drop(columns=['artist', 'album'], errors='ignore')
top_tracks.sample(1)

,id,title,title_short,title_version,link,duration,rank,explicit_lyrics,explicit_content_lyrics,explicit_content_cover,...,album_tracklist,artist_id,artist_name,artist_link,artist_picture,top_tracklist,album_id,album_title,album_cover,album_tracklist
790,2607339722,Better Days,Better Days,,https://www.deezer.com/track/2607339722,180,217327,False,0,0,...,https://api.deezer.com/album/530739702/tracks,4644424,Ryan Ellis,https://www.deezer.com/artist/4644424,https://api.deezer.com/artist/4644424/image,https://api.deezer.com/artist/4644424/top?limi...,530739702,Better Days,https://api.deezer.com/album/530739702/image,https://api.deezer.com/album/530739702/tracks


In [57]:
# deduplicated artist list
artist_list = top_tracks['artist_id'].unique().tolist()
len(artist_list)

1716

In [ ]:
# changed, rewrite this to get the artists from top tracks and get their number of fans
def get_artist_data() -> pd.DataFrame:
    try:
        result = []
        for artist_id in artist_list:
            response = session.get(f'{url}/artist/{artist_id}', timeout=15)
            response.raise_for_status()
            body = response.json() # the whole response
            if "error" in body:
                raise DeezerError('error in artists data body')
            result.append(body)
        return result
    except requests.RequestException as e:
            print("request failed:", e)
            raise
    except DeezerError as e:
            print("deezer error:", e)
            raise

artists_data = get_artist_data()

In [65]:
artists_df = pd.DataFrame(artists_data)

# DATA VALIDATION

add checks for nulls, missing genres, artist names, etc

to validate:
- genres
    - not much to do here. check for null data
- top_tracks
    - handle nulls, less than 100 tracks in genre
- artists_df
    - handle null number of fans, missing data in join

# UPLOAD TO S3

In [ ]:
# top_tracks_upload_df = top_tracks
# artists_upload_df = artists_df

# add all this to main so that it's a python script and package